# Problem 01: Smart Thermostat & Invariant Encapsulation Engine

## Class Design

The solution contains one class:

- `SmartThermostat`
  - Stores the device ID.
  - Encapsulates the current temperature.
  - Encapsulates the target temperature.
  - Provides methods for Fahrenheit conversion and thermostat mode detection.

The temperature attributes are kept private using double underscores. The
`@property` and `@setter` decorators provide controlled access to these
attributes while allowing validation before modifying the internal state.

## Encapsulation Rationale

Direct modification of temperature values is avoided because a thermostat
should maintain valid temperature invariants.

The target temperature setter accepts values only in the range:

10.0°C <= target_temp <= 35.0°C

Invalid target temperatures are rejected without changing the existing target.

The current temperature setter also prevents physically impossible values below
absolute zero (-273.15°C).

## Design Trade-offs

Using properties adds a small amount of code compared with directly exposing
attributes, but it ensures that validation remains centralized. This prevents
invalid state from being introduced through external code.

The device ID is stored privately because it represents internal object state,
while the public properties provide controlled access to temperature values.

In [1]:
class SmartThermostat:
    """Represent a smart thermostat with validated temperature state."""

    def __init__(
        self,
        device_id: str,
        initial_current_temp: float,
        initial_target_temp: float,
    ) -> None:
        """Initialize a smart thermostat."""
        self.__device_id = device_id

        if initial_current_temp < -273.15:
            raise ValueError(
                "Current temperature cannot be below absolute zero."
            )

        if not 10.0 <= initial_target_temp <= 35.0:
            raise ValueError("Invalid Target Temperature")

        self.__current_temp = float(initial_current_temp)
        self.__target_temp = float(initial_target_temp)

    @property
    def current_temp(self) -> float:
        """Return the current temperature in Celsius."""
        return self.__current_temp

    @current_temp.setter
    def current_temp(self, temp: float) -> None:
        """Update current temperature after validating absolute zero."""
        if temp < -273.15:
            print("Error: Invalid Current Temperature")
            return

        self.__current_temp = float(temp)
        print(f"Current Updated: {self.__current_temp:.2f}C")

    @property
    def target_temp(self) -> float:
        """Return the target temperature in Celsius."""
        return self.__target_temp

    @target_temp.setter
    def target_temp(self, temp: float) -> None:
        """Update target temperature if it is within the allowed range."""
        if not 10.0 <= temp <= 35.0:
            print("Error: Invalid Target Temperature")
            return

        self.__target_temp = float(temp)
        print(f"Target Updated: {self.__target_temp:.2f}C")

    @staticmethod
    def to_fahrenheit(celsius: float) -> float:
        """Convert Celsius to Fahrenheit rounded to two decimals."""
        return round((celsius * 9 / 5) + 32, 2)

    def get_mode(self) -> str:
        """Return the thermostat operating mode."""
        if self.__current_temp < self.__target_temp:
            return "HEATING"

        if self.__current_temp > self.__target_temp:
            return "COOLING"

        return "IDLE"

    def __str__(self) -> str:
        """Return a human-readable thermostat status."""
        current_f = self.to_fahrenheit(self.__current_temp)

        return (
            f"Thermostat[{self.__device_id}] "
            f"Current: {self.__current_temp:.2f}C "
            f"({current_f:.2f}F) | "
            f"Target: {self.__target_temp:.2f}C | "
            f"Mode: {self.get_mode()}"
        )

In [2]:
thermostat = SmartThermostat("TH101", 18.5, 22.0)

print(thermostat)

thermostat.target_temp = 25.0
thermostat.target_temp = 40.0
thermostat.current_temp = 25.0

print(thermostat)

Thermostat[TH101] Current: 18.50C (65.30F) | Target: 22.00C | Mode: HEATING
Target Updated: 25.00C
Error: Invalid Target Temperature
Current Updated: 25.00C
Thermostat[TH101] Current: 25.00C (77.00F) | Target: 25.00C | Mode: IDLE


In [3]:
# Cooling mode
cooling_thermostat = SmartThermostat("TH102", 30.0, 25.0)
assert cooling_thermostat.get_mode() == "COOLING"

# Idle mode
idle_thermostat = SmartThermostat("TH103", 20.0, 20.0)
assert idle_thermostat.get_mode() == "IDLE"

# Fahrenheit conversion
assert SmartThermostat.to_fahrenheit(0) == 32.0
assert SmartThermostat.to_fahrenheit(100) == 212.0

# Invalid target should not change the previous value
thermostat.target_temp = 50.0
assert thermostat.target_temp == 25.0

print("All Problem 01 edge-case tests passed.")

Error: Invalid Target Temperature
All Problem 01 edge-case tests passed.


# Problem 02: Library Media Hierarchy & Polymorphic Overdue Engine

## Class Hierarchy

The inheritance hierarchy is:

LibraryItem
├── Book
├── Audiobook
└── DVD

`LibraryItem` stores the common attributes:

- `item_id`
- `title`
- `max_days`

It also provides a default `calculate_fine()` implementation.

Each specialized media class inherits from `LibraryItem` and overrides
`calculate_fine()` according to its own overdue policy.

## Polymorphism

`CirculationDesk` stores all media objects in a single heterogeneous list.
It does not need to determine whether an object is a `Book`, `Audiobook`, or
`DVD`.

Instead, it simply calls:

    item.calculate_fine(days_late)

Python dynamically dispatches the call to the appropriate subclass method.

## Encapsulation Rationale

The attributes are kept as object state and initialized through constructors.
Behavior related to overdue calculation is placed inside the relevant class,
rather than using external conditional logic.

## Design Trade-offs

Inheritance avoids repeating common item information and allows specialized
fine rules to remain inside their corresponding classes.

The trade-off is that the hierarchy is appropriate only while the media types
share a meaningful common abstraction. Adding many unrelated item types could
make a larger inheritance hierarchy harder to maintain.

In [4]:
from typing import List


class LibraryItem:
    """Represent a generic library item."""

    def __init__(self, item_id: str, title: str, max_days: int) -> None:
        """Initialize a library item."""
        self.item_id = item_id
        self.title = title
        self.max_days = max_days

    def calculate_fine(self, days_late: int) -> float:
        """Calculate the default overdue fine."""
        return 0.50 * days_late

    def description(self) -> str:
        """Return a basic description of the item."""
        return f"{self.item_id} {self.title}"


class Book(LibraryItem):
    """Represent a book borrowed from the library."""

    def __init__(self, item_id: str, title: str) -> None:
        """Initialize a book."""
        super().__init__(item_id, title, max_days=14)

    def calculate_fine(self, days_late: int) -> float:
        """Calculate the book overdue fine."""
        return 1.00 * days_late

    def description(self) -> str:
        """Return a book description."""
        return f"{self.item_id} {self.title} (Book)"


class Audiobook(LibraryItem):
    """Represent an audiobook borrowed from the library."""

    def __init__(
        self,
        item_id: str,
        title: str,
        duration_hours: float,
    ) -> None:
        """Initialize an audiobook."""
        super().__init__(item_id, title, max_days=7)
        self.duration_hours = duration_hours

    def calculate_fine(self, days_late: int) -> float:
        """Calculate the audiobook overdue fine."""
        return 1.50 * days_late

    def description(self) -> str:
        """Return an audiobook description."""
        return (
            f"{self.item_id} {self.title} "
            f"(Audiobook, {self.duration_hours:g} hrs)"
        )


class DVD(LibraryItem):
    """Represent a DVD borrowed from the library."""

    def __init__(
        self,
        item_id: str,
        title: str,
        director: str,
    ) -> None:
        """Initialize a DVD."""
        super().__init__(item_id, title, max_days=3)
        self.director = director

    def calculate_fine(self, days_late: int) -> float:
        """Calculate the DVD overdue fine including surcharge."""
        fine = 2.00 * days_late

        if days_late > 5:
            fine += 5.00

        return fine

    def description(self) -> str:
        """Return a DVD description."""
        return (
            f"{self.item_id} {self.title} "
            f"(DVD, Dir: {self.director})"
        )


class CirculationDesk:
    """Manage library items and calculate total overdue fines."""

    def __init__(self, items: List[LibraryItem]) -> None:
        """Initialize the circulation desk."""
        self.items = items

    def calculate_total_fines(self, overdue_days: List[int]) -> float:
        """Calculate total fines using polymorphic dispatch."""
        total = 0.0

        for item, days_late in zip(self.items, overdue_days):
            total += item.calculate_fine(days_late)

        return total

    def print_fines(self, overdue_days: List[int]) -> None:
        """Print individual and total overdue fines."""
        total = 0.0

        for item, days_late in zip(self.items, overdue_days):
            fine = item.calculate_fine(days_late)
            total += fine

            print(
                f"{item.description()}: "
                f"Fine = ${fine:.2f}"
            )

        print(f"Total Outstanding Fines: ${total:.2f}")

In [5]:
items = [
    Book("B101", "PythonDSA"),
    Audiobook("A201", "CleanCode", 8.5),
    DVD("D301", "Inception", "Nolan"),
]

overdue_days = [5, 4, 6]

desk = CirculationDesk(items)
desk.print_fines(overdue_days)

B101 PythonDSA (Book): Fine = $5.00
A201 CleanCode (Audiobook, 8.5 hrs): Fine = $6.00
D301 Inception (DVD, Dir: Nolan): Fine = $17.00
Total Outstanding Fines: $28.00


In [6]:
# No overdue days
assert Book("B1", "DSA").calculate_fine(0) == 0.0
assert Audiobook("A1", "Python", 5.0).calculate_fine(0) == 0.0
assert DVD("D1", "Movie", "Director").calculate_fine(0) == 0.0

# DVD surcharge boundary
dvd = DVD("D2", "Movie", "Director")

assert dvd.calculate_fine(5) == 10.0
assert dvd.calculate_fine(6) == 17.0

# Verify polymorphic behavior
assert items[0].calculate_fine(5) == 5.0
assert items[1].calculate_fine(4) == 6.0

print("All Problem 02 edge-case tests passed.")

All Problem 02 edge-case tests passed.


# Problem 03: Multi-Channel Notification Gateway

## Class Hierarchy

The design uses an Abstract Base Class:

NotificationService
├── EmailService
├── SMSService
└── WebhookService

`NotificationService` defines the common interface:

- `send(recipient, message)`
- `get_cost(message)`

These methods are abstract, so every concrete notification service must
provide its own implementation.

## Abstraction

The dispatcher does not need to know how an email, SMS, or webhook is
implemented. It works against the abstract `NotificationService` contract.

This keeps the alert-dispatching logic decoupled from individual providers.

## Polymorphism

`AlertDispatcher` stores different notification objects in one list and
invokes the same methods on every object:

    service.send(...)
    service.get_cost(...)

The appropriate subclass implementation is selected dynamically.

## Design Trade-offs

An abstract base class introduces a formal contract and makes incorrect
subclasses harder to create accidentally. The trade-off is that every new
notification provider must implement all abstract methods.

The SMS implementation truncates messages longer than 160 characters to
157 characters plus `...`, as required by the specification. Its cost is
calculated from the original message length using 160-character segments.

In [7]:
from abc import ABC, abstractmethod
from math import ceil
from typing import List


class NotificationService(ABC):
    """Abstract interface for notification services."""

    @abstractmethod
    def send(self, recipient: str, message: str) -> str:
        """Send a notification to a recipient."""
        pass

    @abstractmethod
    def get_cost(self, message: str) -> float:
        """Return the cost of sending a message."""
        pass


class EmailService(NotificationService):
    """Provide email-based notifications."""

    def __init__(self, sender_email: str) -> None:
        """Initialize the email service."""
        self.sender_email = sender_email

    def send(self, recipient: str, message: str) -> str:
        """Return the email delivery message."""
        return (
            f"Email sent to {recipient} via {self.sender_email}: "
            f"{message}"
        )

    def get_cost(self, message: str) -> float:
        """Return the fixed email cost."""
        return 0.01


class SMSService(NotificationService):
    """Provide SMS-based notifications."""

    def __init__(self, phone_number: str) -> None:
        """Initialize the SMS service."""
        self.phone_number = phone_number

    def send(self, recipient: str, message: str) -> str:
        """Return the SMS delivery message with length enforcement."""
        if len(message) > 160:
            message = message[:157] + "..."

        return f"SMS sent to {recipient}: {message}"

    def get_cost(self, message: str) -> float:
        """Return the SMS cost based on 160-character segments."""
        segments = max(1, ceil(len(message) / 160))
        return segments * 0.05


class WebhookService(NotificationService):
    """Provide webhook-based notifications."""

    def __init__(self, endpoint_url: str) -> None:
        """Initialize the webhook service."""
        self.endpoint_url = endpoint_url

    def send(self, recipient: str, message: str) -> str:
        """Return the webhook delivery representation."""
        return (
            f"POST payload to {self.endpoint_url} for "
            f"{recipient}: {message}"
        )

    def get_cost(self, message: str) -> float:
        """Return the free webhook cost."""
        return 0.00


class AlertDispatcher:
    """Broadcast alerts through multiple notification services."""

    def __init__(self, services: List[NotificationService]) -> None:
        """Initialize the dispatcher."""
        self.services = services

    def broadcast(self, recipient: str, message: str) -> float:
        """Send an alert through all services and return total cost."""
        total_cost = 0.0

        for service in self.services:
            print(service.send(recipient, message))
            cost = service.get_cost(message)
            print(f"Cost: ${cost:.2f}")
            total_cost += cost

        print(f"Total Broadcast Cost: ${total_cost:.2f}")

        return total_cost

In [8]:
recipient = "AdminUser"
message = "CRITICAL: High CPU utilization detected on server node 04."

services = [
    EmailService("alerts@cloud.com"),
    SMSService("+1999888777"),
    WebhookService("https://hooks.slack.com/services/alert"),
]

dispatcher = AlertDispatcher(services)

total_cost = dispatcher.broadcast(recipient, message)

assert round(total_cost, 2) == 0.06

Email sent to AdminUser via alerts@cloud.com: CRITICAL: High CPU utilization detected on server node 04.
Cost: $0.01
SMS sent to AdminUser: CRITICAL: High CPU utilization detected on server node 04.
Cost: $0.05
POST payload to https://hooks.slack.com/services/alert for AdminUser: CRITICAL: High CPU utilization detected on server node 04.
Cost: $0.00
Total Broadcast Cost: $0.06


In [9]:
sms = SMSService("+123456789")

short_message = "Hello"
assert sms.send("User1", short_message) == "SMS sent to User1: Hello"
assert sms.get_cost(short_message) == 0.05

message_160 = "A" * 160
message_161 = "A" * 161

assert sms.get_cost(message_160) == 0.05
assert sms.get_cost(message_161) == 0.10

truncated = sms.send("User1", message_161)

assert len(truncated.split(": ", 1)[1]) == 160
assert truncated.endswith("...")

print("All Problem 03 edge-case tests passed.")

All Problem 03 edge-case tests passed.


# Problem 04: Exact Rational Fraction Engine & Operator Overloading

## Class Design

The solution uses one immutable `Rational` class.

Each object stores:

- numerator
- denominator

The constructor enforces three invariants:

1. The denominator cannot be zero.
2. The denominator must always be positive.
3. The fraction must always be stored in reduced form.

For example:

    Rational(1, -2)

is internally represented as:

    -1/2

## Operator Overloading

The class overloads:

- `__add__` for addition
- `__sub__` for subtraction
- `__mul__` for multiplication
- `__truediv__` for division
- `__eq__` for equality
- `__lt__` for less-than comparison
- `__str__` for readable output
- `__float__` for conversion to floating point

Arithmetic operations create and return a new `Rational` object instead of
modifying the existing object.

## Immutability

No setter is provided for numerator or denominator. Once constructed, a
Rational object's mathematical value is not modified by arithmetic operations.

This prevents accidental corruption of the reduced-fraction invariant.

## Design Trade-offs

Exact integer arithmetic avoids floating-point precision errors. The trade-off
is that fractions require numerator and denominator management and GCD
normalization.

A small coercion helper allows integer operands to participate naturally in
arithmetic while still preserving exact rational representation.

In [10]:
from math import gcd
from typing import Union


Number = Union[int, "Rational"]


class Rational:
    """Represent an immutable rational number."""

    def __init__(self, numerator: int, denominator: int) -> None:
        """Initialize and normalize a rational number."""
        if denominator == 0:
            raise ZeroDivisionError("Denominator cannot be zero.")

        if denominator < 0:
            numerator = -numerator
            denominator = -denominator

        common_divisor = gcd(abs(numerator), abs(denominator))

        self.__numerator = numerator // common_divisor
        self.__denominator = denominator // common_divisor

    @property
    def numerator(self) -> int:
        """Return the normalized numerator."""
        return self.__numerator

    @property
    def denominator(self) -> int:
        """Return the normalized positive denominator."""
        return self.__denominator

    @staticmethod
    def _coerce(other: Number) -> "Rational":
        """Convert an integer or Rational to Rational."""
        if isinstance(other, Rational):
            return other

        if isinstance(other, int):
            return Rational(other, 1)

        return NotImplemented

    def __add__(self, other: Number) -> "Rational":
        """Return the exact sum of two rational values."""
        other = self._coerce(other)

        if other is NotImplemented:
            return NotImplemented

        numerator = (
            self.__numerator * other.denominator
            + other.numerator * self.__denominator
        )
        denominator = self.__denominator * other.denominator

        return Rational(numerator, denominator)

    def __sub__(self, other: Number) -> "Rational":
        """Return the exact difference of two rational values."""
        other = self._coerce(other)

        if other is NotImplemented:
            return NotImplemented

        numerator = (
            self.__numerator * other.denominator
            - other.numerator * self.__denominator
        )
        denominator = self.__denominator * other.denominator

        return Rational(numerator, denominator)

    def __mul__(self, other: Number) -> "Rational":
        """Return the exact product of two rational values."""
        other = self._coerce(other)

        if other is NotImplemented:
            return NotImplemented

        return Rational(
            self.__numerator * other.numerator,
            self.__denominator * other.denominator,
        )

    def __truediv__(self, other: Number) -> "Rational":
        """Return the exact quotient of two rational values."""
        other = self._coerce(other)

        if other is NotImplemented:
            return NotImplemented

        if other.numerator == 0:
            raise ZeroDivisionError("Cannot divide by zero.")

        return Rational(
            self.__numerator * other.denominator,
            self.__denominator * other.numerator,
        )

    def __eq__(self, other: object) -> bool:
        """Return whether two rational values are equal."""
        if isinstance(other, int):
            other = Rational(other, 1)

        if not isinstance(other, Rational):
            return NotImplemented

        return (
            self.__numerator * other.denominator
            == other.numerator * self.__denominator
        )

    def __lt__(self, other: Number) -> bool:
        """Return whether this rational is smaller than another."""
        other = self._coerce(other)

        if other is NotImplemented:
            return NotImplemented

        return (
            self.__numerator * other.denominator
            < other.numerator * self.__denominator
        )

    def __str__(self) -> str:
        """Return the simplified fraction as a string."""
        if self.__denominator == 1:
            return str(self.__numerator)

        return f"{self.__numerator}/{self.__denominator}"

    def __float__(self) -> float:
        """Return the floating-point representation."""
        return self.__numerator / self.__denominator

In [11]:
f1 = Rational(1, 3)
f2 = Rational(1, 6)

print(f"f1 = {f1}, f2 = {f2}")
print(f"f1 + f2 = {f1 + f2} (float: {float(f1 + f2):.4f})")
print(f"f1 - f2 = {f1 - f2}")
print(f"f1 * f2 = {f1 * f2}")
print(f"f1 / f2 = {f1 / f2}")
print(f"f1 > f2: {f1 > f2}")
print(f"f1 == f2: {f1 == f2}")

f1 = 1/3, f2 = 1/6
f1 + f2 = 1/2 (float: 0.5000)
f1 - f2 = 1/6
f1 * f2 = 1/18
f1 / f2 = 2
f1 > f2: True
f1 == f2: False


In [12]:
# Negative denominator normalization
negative = Rational(1, -2)
assert str(negative) == "-1/2"

# Automatic reduction
reduced = Rational(10, 20)
assert str(reduced) == "1/2"

# Zero numerator
zero = Rational(0, 25)
assert str(zero) == "0"

# Exact arithmetic
assert str(Rational(1, 3) + Rational(1, 6)) == "1/2"

# Division by zero
try:
    Rational(1, 2) / Rational(0, 1)
except ZeroDivisionError:
    print("Division by zero correctly rejected.")

# Zero denominator
try:
    Rational(1, 0)
except ZeroDivisionError:
    print("Zero denominator correctly rejected.")

print("All Problem 04 edge-case tests passed.")

Division by zero correctly rejected.
Zero denominator correctly rejected.
All Problem 04 edge-case tests passed.


# Problem 05: Multi-Entity E-Commerce Fulfillment & State Machine Engine

## Class Design

The system contains four classes:

- `Product`
- `Customer`
- `Order`
- `ECommerceEngine`

The classes use composition rather than inheritance because they represent
different entities.

An `Order` contains:

- a `Customer`
- a collection of `Product` objects with quantities
- an order status

The `ECommerceEngine` aggregates products, customers, and orders and manages
the workflow.

## Order State Machine

The valid order lifecycle is:

CREATED -> PAID -> SHIPPED -> DELIVERED

Cancellation is allowed from:

CREATED -> CANCELLED
PAID -> CANCELLED

Once an order reaches `SHIPPED`, `DELIVERED`, or `CANCELLED`, invalid
transitions are rejected.

## Invariant Guarding

When an order is created, the engine checks product stock before reserving
anything.

If sufficient stock exists, the requested quantities are reserved immediately.

During payment:

- The customer wallet is checked.
- If sufficient funds exist, the amount is deducted and the order becomes
  `PAID`.
- Otherwise, the order is cancelled and all reserved stock is restored.

During cancellation:

- Reserved stock is restored.
- If the order was already paid, the amount is refunded to the customer.
- The order becomes `CANCELLED`.

## Design Trade-offs

Composition is used because Product, Customer, and Order have different
responsibilities and do not naturally form an inheritance hierarchy.

The state transition logic is centralized inside `Order`, while the
`ECommerceEngine` coordinates entities and performs business operations.

This separation makes illegal transitions easier to control and keeps the
engine from directly manipulating an order's status without validation.

In [13]:
from typing import Dict, List


class Product:
    """Represent a product available in the e-commerce system."""

    def __init__(
        self,
        product_id: str,
        name: str,
        price: float,
        stock: int,
    ) -> None:
        """Initialize a product."""
        if price < 0:
            raise ValueError("Product price cannot be negative.")

        if stock < 0:
            raise ValueError("Product stock cannot be negative.")

        self.product_id = product_id
        self.name = name
        self.price = float(price)
        self.stock = stock


class Customer:
    """Represent a customer with a wallet balance."""

    def __init__(
        self,
        customer_id: str,
        name: str,
        wallet_balance: float,
    ) -> None:
        """Initialize a customer."""
        if wallet_balance < 0:
            raise ValueError("Wallet balance cannot be negative.")

        self.customer_id = customer_id
        self.name = name
        self.wallet_balance = float(wallet_balance)


class Order:
    """Represent an order and enforce its state transitions."""

    VALID_TRANSITIONS = {
        "CREATED": {"PAID", "CANCELLED"},
        "PAID": {"SHIPPED", "CANCELLED"},
        "SHIPPED": {"DELIVERED"},
        "DELIVERED": set(),
        "CANCELLED": set(),
    }

    def __init__(
        self,
        order_id: str,
        customer: Customer,
        items: Dict[Product, int],
    ) -> None:
        """Initialize an order in the CREATED state."""
        self.order_id = order_id
        self.customer = customer
        self.items = items
        self.status = "CREATED"

    def total(self) -> float:
        """Calculate the total price of the order."""
        return sum(
            product.price * quantity
            for product, quantity in self.items.items()
        )

    def transition_to(self, new_status: str) -> bool:
        """Transition to a valid next state."""
        if new_status not in self.VALID_TRANSITIONS[self.status]:
            return False

        self.status = new_status
        return True


class ECommerceEngine:
    """Coordinate products, customers, orders, and order workflows."""

    def __init__(self) -> None:
        """Initialize an empty e-commerce engine."""
        self.products: Dict[str, Product] = {}
        self.customers: Dict[str, Customer] = {}
        self.orders: Dict[str, Order] = {}

    def add_product(self, product: Product) -> None:
        """Add a product to the product catalog."""
        self.products[product.product_id] = product

    def add_customer(self, customer: Customer) -> None:
        """Add a customer to the system."""
        self.customers[customer.customer_id] = customer

    def create_order(
        self,
        order_id: str,
        customer_id: str,
        item_requests: Dict[str, int],
    ) -> bool:
        """Create an order after validating and reserving stock."""
        if order_id in self.orders:
            print(f"Error: Order {order_id} already exists.")
            return False

        if customer_id not in self.customers:
            print("Error: Customer not found.")
            return False

        # Validate quantities and product availability first.
        for product_id, quantity in item_requests.items():
            if product_id not in self.products:
                print(f"Error: Product {product_id} not found.")
                return False

            if quantity <= 0:
                print("Error: Quantity must be positive.")
                return False

            if self.products[product_id].stock < quantity:
                print(
                    f"Error: Insufficient stock for "
                    f"{self.products[product_id].name}."
                )
                return False

        # Reserve stock only after all products pass validation.
        items: Dict[Product, int] = {}

        for product_id, quantity in item_requests.items():
            product = self.products[product_id]
            product.stock -= quantity
            items[product] = quantity

        order = Order(
            order_id,
            self.customers[customer_id],
            items,
        )

        self.orders[order_id] = order

        print(
            f"Order {order_id} Created: "
            f"Total = ${order.total():.2f}"
        )

        return True

    def pay_order(self, order_id: str) -> bool:
        """Pay for an order or cancel it when funds are insufficient."""
        if order_id not in self.orders:
            print("Error: Order not found.")
            return False

        order = self.orders[order_id]

        if order.status != "CREATED":
            print(
                f"Error: Cannot pay in {order.status} state."
            )
            return False

        total = order.total()

        if order.customer.wallet_balance < total:
            self._restore_stock(order)
            order.transition_to("CANCELLED")

            print("Payment Failed: Insufficient Balance")
            return False

        order.customer.wallet_balance -= total
        order.transition_to("PAID")

        print(f"Order {order_id} Paid Successfully.")
        print(
            f"{order.customer.name} Remaining Balance: "
            f"${order.customer.wallet_balance:.2f}"
        )

        return True

    def ship_order(self, order_id: str) -> bool:
        """Move an order from PAID to SHIPPED."""
        if order_id not in self.orders:
            print("Error: Order not found.")
            return False

        order = self.orders[order_id]

        if not order.transition_to("SHIPPED"):
            print(
                f"Error: Cannot ship in {order.status} state."
            )
            return False

        print(
            f"Order {order_id} Status: {order.status}"
        )

        return True

    def deliver_order(self, order_id: str) -> bool:
        """Move an order from SHIPPED to DELIVERED."""
        if order_id not in self.orders:
            print("Error: Order not found.")
            return False

        order = self.orders[order_id]

        if not order.transition_to("DELIVERED"):
            print(
                f"Error: Cannot deliver in {order.status} state."
            )
            return False

        print(
            f"Order {order_id} Status: {order.status}"
        )

        return True

    def cancel_order(self, order_id: str) -> bool:
        """Cancel an order and restore stock/refund payment if required."""
        if order_id not in self.orders:
            print("Error: Order not found.")
            return False

        order = self.orders[order_id]

        if order.status not in {"CREATED", "PAID"}:
            print(
                f"Error: Cannot cancel in {order.status} state."
            )
            return False

        if order.status == "PAID":
            order.customer.wallet_balance += order.total()

        self._restore_stock(order)
        order.transition_to("CANCELLED")

        print(f"Order {order_id} Status: CANCELLED")

        return True

    @staticmethod
    def _restore_stock(order: Order) -> None:
        """Restore reserved quantities to their products."""
        for product, quantity in order.items.items():
            product.stock += quantity

In [14]:
engine = ECommerceEngine()

engine.add_product(
    Product("P101", "Laptop", 1000.00, 5)
)

engine.add_product(
    Product("P102", "Mouse", 50.00, 10)
)

engine.add_customer(
    Customer("C01", "Alice", 1500.00)
)

engine.create_order(
    "O1",
    "C01",
    {
        "P101": 1,
        "P102": 2,
    },
)

engine.pay_order("O1")
engine.ship_order("O1")
engine.deliver_order("O1")
engine.cancel_order("O1")

Order O1 Created: Total = $1100.00
Order O1 Paid Successfully.
Alice Remaining Balance: $400.00
Order O1 Status: SHIPPED
Order O1 Status: DELIVERED
Error: Cannot cancel in DELIVERED state.


False

In [15]:
# ---------------------------------------------------------
# Test 1: Insufficient balance
# ---------------------------------------------------------

engine2 = ECommerceEngine()

product = Product("P1", "Keyboard", 500.00, 5)
customer = Customer("C1", "Bob", 100.00)

engine2.add_product(product)
engine2.add_customer(customer)

engine2.create_order(
    "O2",
    "C1",
    {"P1": 2},
)

assert product.stock == 3

engine2.pay_order("O2")

assert engine2.orders["O2"].status == "CANCELLED"
assert product.stock == 5
assert customer.wallet_balance == 100.00

print("Insufficient-balance test passed.")


# ---------------------------------------------------------
# Test 2: Successful cancellation after payment
# ---------------------------------------------------------

engine3 = ECommerceEngine()

product3 = Product("P3", "Monitor", 300.00, 4)
customer3 = Customer("C3", "Charlie", 1000.00)

engine3.add_product(product3)
engine3.add_customer(customer3)

engine3.create_order(
    "O3",
    "C3",
    {"P3": 1},
)

assert product3.stock == 3

engine3.pay_order("O3")

assert customer3.wallet_balance == 700.00

engine3.cancel_order("O3")

assert engine3.orders["O3"].status == "CANCELLED"
assert product3.stock == 4
assert customer3.wallet_balance == 1000.00

print("Paid-order cancellation test passed.")


# ---------------------------------------------------------
# Test 3: Illegal transition
# ---------------------------------------------------------

engine4 = ECommerceEngine()

product4 = Product("P4", "Phone", 800.00, 2)
customer4 = Customer("C4", "David", 1000.00)

engine4.add_product(product4)
engine4.add_customer(customer4)

engine4.create_order(
    "O4",
    "C4",
    {"P4": 1},
)

# Cannot deliver directly from CREATED.
result = engine4.deliver_order("O4")

assert result is False
assert engine4.orders["O4"].status == "CREATED"

print("Illegal-transition test passed.")


# ---------------------------------------------------------
# Test 4: Insufficient stock
# ---------------------------------------------------------

engine5 = ECommerceEngine()

product5 = Product("P5", "Tablet", 600.00, 2)
customer5 = Customer("C5", "Eva", 2000.00)

engine5.add_product(product5)
engine5.add_customer(customer5)

result = engine5.create_order(
    "O5",
    "C5",
    {"P5": 5},
)

assert result is False
assert product5.stock == 2
assert "O5" not in engine5.orders

print("Insufficient-stock test passed.")

print("All Problem 05 edge-case tests passed.")

Order O2 Created: Total = $1000.00
Payment Failed: Insufficient Balance
Insufficient-balance test passed.
Order O3 Created: Total = $300.00
Order O3 Paid Successfully.
Charlie Remaining Balance: $700.00
Order O3 Status: CANCELLED
Paid-order cancellation test passed.
Order O4 Created: Total = $800.00
Error: Cannot deliver in CREATED state.
Illegal-transition test passed.
Error: Insufficient stock for Tablet.
Insufficient-stock test passed.
All Problem 05 edge-case tests passed.
